# Golf Swing Sequence Pipeline

This notebook orchestrates tasks **A–H**: centralized configs, metadata standardization, swing-window trimming, MediaPipe pose extraction, spatial/temporal normalization, feature engineering, augmentation, and dataset export. Legacy `Balanced_Dataset` routines have been retired in favor of the reproducible `processed_videos` pipeline documented here.

In [1]:
# --- Imports & deterministic setup (Task A) ---
import json
import warnings
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:  # pragma: no cover - tqdm is optional
    def tqdm(iterable, **kwargs):
        return iterable

import config
print("config OK")

# Force reload pipeline modules to pick up any changes
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('pipeline'):
        del sys.modules[mod_name]

from pipeline import metadata_utils
from pipeline.feature_engineering import FeatureEngineer
from pipeline.pose_processing import PoseProcessor
from pipeline.video_processing import VideoProcessor

# Verify reset() method exists
assert hasattr(PoseProcessor, 'reset'), "PoseProcessor.reset() not found - module not reloaded properly"
print("PoseProcessor.reset() verified ✓")

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)

print(f"Working directory: {config.BASE_DIR}")
print(f"Data root exists: {config.DATA_ROOT.exists()} | Video output: {config.OUTPUT_ROOT}")

C:\Users\Lenovo\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


config OK
PoseProcessor.reset() verified ✓
Working directory: D:\DAI_HOC\THI\DATASTORM 2025\v2
Data root exists: True | Video output: D:\DAI_HOC\THI\DATASTORM 2025\v2\processed_videos


## Task A – Reproducibility & Config Audits
- Load every configurable quantity (roots, bands, FPS, stride, padding, thresholds) from `config.py`.
- Standardize folder → label mapping so `Indoor/Band 1-2` becomes `env=indoor`, `band=1_2`.
- Keep notebook logic free of magic paths; all references go through the centralized config module.

In [2]:
from IPython.display import display

# --- Inspect config-driven mappings and hyper-parameters ---
config_summary = pd.DataFrame(
    [
        ("DATA_ROOT", config.DATA_ROOT),
        ("OUTPUT_ROOT", config.OUTPUT_ROOT),
        ("ENVIRONMENTS", ", ".join(config.ENVIRONMENTS)),
        ("BANDS", ", ".join(config.BANDS)),
        ("TARGET_FPS", config.TARGET_FPS),
        ("N_FRAMES", config.N_FRAMES),
        ("STRIDE", config.STRIDE),
        ("PADDING_MARGIN_FRAMES", config.PADDING_MARGIN_FRAMES),
        ("YOLO_CONF", config.THRESHOLDS["yolo_conf"]),
        ("POSE_VISIBILITY", config.THRESHOLDS["pose_visibility"]),
    ],
    columns=["key", "value"],
)

print("Centralized configuration snapshot:")
display(config_summary)

print("Environment mapping:")
display(pd.DataFrame(config.ENVIRONMENT_FOLDER_MAP.items(), columns=["folder", "env"]))
print("Band mapping:")
display(pd.DataFrame(config.BAND_FOLDER_MAP.items(), columns=["folder", "band"]))

Centralized configuration snapshot:


,key,value
0,DATA_ROOT,D:\DAI_HOC\THI\DATASTORM 2025\v2\Public Test
1,OUTPUT_ROOT,D:\DAI_HOC\THI\DATASTORM 2025\v2\processed_videos
2,ENVIRONMENTS,"indoor, outdoor"
3,BANDS,"1_2, 2_4, 4_6, 6_8, 8_10"
4,TARGET_FPS,30
5,N_FRAMES,100
6,STRIDE,1
7,PADDING_MARGIN_FRAMES,5
8,YOLO_CONF,0.35
9,POSE_VISIBILITY,0.5


Environment mapping:


,folder,env
0,Trong nhà - Indoor,indoor
1,Ngoài trời - Outdoor,outdoor
2,Indoor,indoor
3,Outdoor,outdoor


Band mapping:


,folder,band
0,Band 1-2,1_2
1,Band 2-4,2_4
2,Band 4-6,4_6
3,Band 6-8,6_8
4,Band 8-10,8_10


## Task A – Metadata normalization
We build `metadata.csv` (video_id, env, band, relative path, fps, duration, frame_count, resolution, swing window placeholders). This documents the `processed_videos` pipeline that supersedes the old Balanced_Dataset naming.

In [3]:
# --- Build metadata.csv with standardized labels ---
metadata_path = config.BASE_DIR / "metadata.csv"
metadata_df = metadata_utils.build_metadata_csv(metadata_path)
print(f"Metadata saved to: {metadata_path.relative_to(config.BASE_DIR)} | videos={len(metadata_df)}")

if not metadata_df.empty:
    counts = metadata_df.groupby(["env", "band"]).size().reset_index(name="video_count")
    display(counts.sort_values(["env", "band"]))
else:
    print("No videos detected under DATA_ROOT. Please check the dataset layout.")

Metadata saved to: metadata.csv | videos=50


,env,band,video_count
0,indoor,1_2,6
1,indoor,2_4,4
2,indoor,4_6,6
3,indoor,6_8,6
4,indoor,8_10,4
5,outdoor,1_2,4
6,outdoor,2_4,4
7,outdoor,4_6,6
8,outdoor,6_8,6
9,outdoor,8_10,4


## Tasks B–E – Video cleaning, swing windowing, pose robustness, spatial & temporal normalization
- Resample every clip to `target_fps` before any pose work.
- Use wrist velocity (fallback torso rotation) to isolate a single swing window and store `swing_start_frame`/`swing_end_frame`.
- Rely on one MediaPipe API (Tasks Pose Landmarker) for extraction and interpolation + Savitzky–Golay smoothing.
- Translate mid-hip to the origin, scale by hip width, align hip axis with X, shoulder axis with +Y, and mirror left-handed swings.
- Resample each pose track to `N_FRAMES` continuous frames for downstream sequence models.

In [4]:
# --- Helper utilities for Tasks B–E ---
from pipeline.types import PoseSequence

video_processor = VideoProcessor()
pose_processor = PoseProcessor()
feature_engineer = FeatureEngineer()

print("Initialized processors:")
print(" - VideoProcessor → target fps:", video_processor.target_fps)
print(" - PoseProcessor model:", config.POSE_MODEL_PATH.name)
print(" - FeatureEngineer feature dim:", config.POSE_FEATURE_DIM)

def slice_pose_sequence(sequence: PoseSequence, start: int, end: int) -> PoseSequence:
    'Return a shallow copy of PoseSequence restricted to [start, end).'
    start = max(0, start)
    end = min(len(sequence.data), max(start + 1, end))
    return PoseSequence(
        data=sequence.data[start:end].copy(),
        frame_times=sequence.frame_times[start:end].copy(),
        fps=sequence.fps,
        interpolation_mask=sequence.interpolation_mask[start:end].copy(),
        valid_mask=sequence.valid_mask[start:end].copy(),
    )

Initialized processors:
 - VideoProcessor → target fps: 30
 - PoseProcessor model: pose_landmarker_full.task
 - FeatureEngineer feature dim: 16


## Tasks F–H – Feature engineering, augmentation, and dataset export

**Important Note on Data Leakage Prevention:**
- During `save_sample()`, features are saved to disk but scaler stats are NOT updated
- Scaler fitting happens AFTER splits are assigned, using only train + non-augmented samples
- This ensures validation/test data does NOT influence the normalization parameters

**Pipeline:**
- Per-frame features: normalized keypoints, velocity, acceleration, joint angles, X-factor, hip–shoulder separation, wrist speed.
- Temporal-safe augmentations on pose (Gaussian jitter, ±5% time-warp, temporal jitter, light dropout + interpolation).
- Save tensors as `.npz` with metadata (`env`, `band`, `video_id`, quality metrics) plus interpolation masks.
- **After splits assigned:** Fit z-score scaler on train-only, normalize all saved tensors, and export metadata.

In [5]:
# --- Master pipeline loop for Tasks B–H ---
metadata_updates = []
error_log = []

if metadata_df.empty:
    print("No videos to process – aborting pipeline run.")
else:
    for row in tqdm(metadata_df.itertuples(index=False), total=len(metadata_df), desc="Processing videos"):
        p = Path(str(row.file_path))
        video_path = p if p.is_absolute() else (config.BASE_DIR / p)
        if not video_path.exists():
            warning_msg = f"Missing file: {video_path}"
            warnings.warn(warning_msg)
            metadata_updates.append(
                {
                    "video_id": row.video_id,
                    "swing_start_frame": -1,
                    "swing_end_frame": -1,
                    "trimmed_frame_count": 0,
                    "swing_method": "missing",
                    "low_quality": True,
                    "quality_valid_ratio": 0.0,
                    "quality_mean_visibility": 0.0,
                    "quality_longest_dropout": 0,
                    "processed_sequence_path": None,
                    "interpolation_mask_path": None,
                    "processing_error": warning_msg,
                }
            )
            continue

        clip = None
        payload = None
        quality = None
        swing_window = None
        trimmed_duration = 0.0
        try:
            # Load and resample the full clip
            clip = video_processor.load_and_resample(video_path)
            
            # Extract pose on full clip for swing detection
            pose_processor.reset()  # Ensure clean state for new video
            raw_sequence = pose_processor.extract_sequence(clip.frames, clip.fps)
            
            # Detect swing window from full-clip pose
            swing_window = video_processor.detect_swing_window(raw_sequence)
            
            # ISSUE 1 FIX: Trim FRAMES (not just pose) with padding
            pad = config.PADDING_MARGIN_FRAMES
            start_frame = max(0, swing_window.start_frame - pad)
            end_frame = min(len(clip.frames), swing_window.end_frame + pad)
            trimmed_frames = clip.frames[start_frame:end_frame]
            
            # ISSUE 4 FIX: Check for short sequences
            if len(trimmed_frames) < 3:
                raise RuntimeError(f"Trimmed sequence too short: {len(trimmed_frames)} frames (need at least 3)")
            
            # Reset landmarker before re-extracting (VIDEO mode requires monotonic timestamps)
            pose_processor.reset()
            
            # Re-extract pose on trimmed frames
            raw_trimmed_sequence = pose_processor.extract_sequence(trimmed_frames, clip.fps)
            
            # ISSUE 2 FIX: Compute trimmed_duration BEFORE prepare_sequence() using raw frame times
            if len(raw_trimmed_sequence.frame_times) > 1:
                trimmed_duration = float(raw_trimmed_sequence.frame_times[-1] - raw_trimmed_sequence.frame_times[0])
            else:
                trimmed_duration = 0.0
            
            # Now prepare the sequence (resampling to N_FRAMES happens here)
            processed_sequence, quality = pose_processor.prepare_sequence(raw_trimmed_sequence)
            
            features = feature_engineer.compute_features(processed_sequence)
            payload = feature_engineer.save_sample(
                video_id=str(row.video_id),
                env=str(row.env),
                band=str(row.band),
                features=features,
                interpolation_mask=processed_sequence.interpolation_mask,
                quality=quality,
                metadata={"trimmed_duration_s": trimmed_duration},
            )

            if not quality.low_quality:
                augmentations = feature_engineer.augment_sequence(processed_sequence.data)
                for aug_name, aug_data in augmentations.items():
                    # ISSUE 3 FIX: Create NEW interpolation mask for augmented samples
                    aug_interp_mask = np.zeros_like(processed_sequence.interpolation_mask, dtype=bool)
                    
                    aug_sequence = PoseSequence(
                        data=aug_data,
                        frame_times=processed_sequence.frame_times,
                        fps=processed_sequence.fps,
                        interpolation_mask=aug_interp_mask,
                        valid_mask=processed_sequence.valid_mask,
                    )
                    aug_features = feature_engineer.compute_features(aug_sequence)
                    feature_engineer.save_sample(
                        video_id=str(row.video_id),
                        env=str(row.env),
                        band=str(row.band),
                        features=aug_features,
                        interpolation_mask=aug_interp_mask,
                        quality=quality,
                        metadata={"augmented": True, "strategy": aug_name},
                        augmented_suffix=aug_name,
                    )

            metadata_updates.append(
                {
                    "video_id": row.video_id,
                    "swing_start_frame": int(swing_window.start_frame),
                    "swing_end_frame": int(swing_window.end_frame),
                    "trimmed_frame_count": len(trimmed_frames),
                    "swing_method": swing_window.method,
                    "low_quality": quality.low_quality,
                    "quality_valid_ratio": quality.valid_ratio,
                    "quality_mean_visibility": quality.mean_visibility,
                    "quality_longest_dropout": quality.longest_dropout,
                    "processed_sequence_path": str(payload.sequence_path.relative_to(config.BASE_DIR)) if payload else None,
                    "interpolation_mask_path": str(payload.interpolation_mask_path.relative_to(config.BASE_DIR)) if payload else None,
                    "processing_error": None,
                }
            )
        except Exception as exc:  # pragma: no cover - debugging aid
            warning_msg = f"{row.video_id}: {exc}"
            warnings.warn(warning_msg)
            error_log.append(warning_msg)
            metadata_updates.append(
                {
                    "video_id": str(row.video_id),
                    "swing_start_frame": swing_window.start_frame if swing_window else -1,
                    "swing_end_frame": swing_window.end_frame if swing_window else -1,
                    "trimmed_frame_count": swing_window.num_frames if swing_window else 0,
                    "swing_method": swing_window.method if swing_window else "error",
                    "low_quality": True,
                    "quality_valid_ratio": 0.0,
                    "quality_mean_visibility": 0.0,
                    "quality_longest_dropout": 0,
                    "processed_sequence_path": None,
                    "interpolation_mask_path": None,
                    "processing_error": warning_msg,
                }
            )
        finally:
            if clip is not None:
                del clip

    print(f"Completed processing {len(metadata_updates)} entries. Errors: {len(error_log)}")
    print("\n--- FIXES APPLIED ---")
    print("1. Swing window: Frames are now trimmed (not just pose), then pose re-extracted")
    print("2. Duration: Computed from raw_trimmed_sequence BEFORE resampling")
    print("3. Augmentation masks: Each augmented sample gets a fresh all-False mask")
    print("4. Short sequence check: Raises RuntimeError if < 3 frames")
    print("5. Landmarker reset: pose_processor.reset() called before each extraction")

Processing videos: 100%|██████████| 50/50 [05:11<00:00,  6.22s/it]

Completed processing 50 entries. Errors: 0

--- FIXES APPLIED ---
1. Swing window: Frames are now trimmed (not just pose), then pose re-extracted
2. Duration: Computed from raw_trimmed_sequence BEFORE resampling
3. Augmentation masks: Each augmented sample gets a fresh all-False mask
4. Short sequence check: Raises RuntimeError if < 3 frames
5. Landmarker reset: pose_processor.reset() called before each extraction


## Normalization, scaler export, and dataset splits

**CRITICAL: Scaler Fitting Order to Prevent Data Leakage**

The z-score scaler (mean/std) must be computed ONLY from training data to avoid data leakage:
1. **Split assignment first**: `export_splits()` assigns train/val/test to each video_id
2. **Fit scaler on train only**: `fit_scaler_from_saved_sequences()` reads saved .npz files and updates stats ONLY for:
   - `split == "train"` (exclude val/test)
   - `augmented == False` (exclude augmented versions)
   - `low_quality == False` (optional, exclude poor quality samples)
3. **Finalize and normalize**: Compute mean/std, then apply to ALL sequences (train/val/test)

**Why this matters**: If we include val/test samples when computing mean/std, we leak information about the validation/test distributions into the normalization process. This makes evaluation optimistically biased because the model sees normalized features that were influenced by test data statistics.

In [6]:
# --- Finalize scaler, normalize tensors, export splits/metadata ---
if metadata_updates:
    updates_df = pd.DataFrame(metadata_updates).set_index("video_id")
    metadata_df = metadata_df.set_index("video_id")
    metadata_df.update(updates_df)
    metadata_df = metadata_df.reset_index()
    metadata_df.to_csv(metadata_path, index=False)
    print(f"Updated metadata with swing + quality fields → {metadata_path.relative_to(config.BASE_DIR)}")
else:
    print("No metadata updates recorded.")

if feature_engineer.samples:
    # STEP 1: Assign splits (this updates sample_records with 'split' field)
    split_path = feature_engineer.export_splits()
    print(f"✓ Splits assigned and saved to {split_path.relative_to(config.BASE_DIR)}")
    
    # STEP 2: Fit scaler ONLY on train split, non-augmented samples
    # This prevents data leakage - val/test must not influence normalization
    print("\n--- Fitting z-score scaler (train only, non-augmented) ---")
    scaler_fit_count = feature_engineer.fit_scaler_from_saved_sequences(exclude_low_quality=True)
    print(f"✓ Scaler fit on {scaler_fit_count} train samples")
    print(f"  Total sample records: {len(feature_engineer.sample_records)}")
    print(f"  Scaler stats count: {feature_engineer.stats.count} feature vectors")
    
    # STEP 3: Finalize scaler (compute mean/std and save JSON)
    mean, std = feature_engineer.finalize_scaler()
    print(f"✓ Scaler saved to {config.FINAL_FEATURE_SUBDIR}/{config.SCALER_FILENAME}")
    print(f"  Mean shape: {mean.shape}, Std shape: {std.shape}")
    
    # STEP 4: Apply normalization to ALL sequences (train/val/test)
    feature_engineer.normalize_saved_sequences(mean, std)
    print(f"✓ Normalized all saved sequences with train-only scaler")
    
    # STEP 5: Export metadata
    processed_metadata_path = feature_engineer.export_metadata()
    print(f"✓ Processed sample metadata: {processed_metadata_path.relative_to(config.BASE_DIR)}")
    
    # STEP 6: Update main metadata with split assignments
    with open(split_path, "r", encoding="utf-8") as f:
        split_map = json.load(f)
    metadata_df["split"] = metadata_df["video_id"].map(split_map)
    metadata_df.to_csv(metadata_path, index=False)
    print(f"✓ Main metadata updated with split assignments")
    
    print("\n--- Scaler Fitting Summary ---")
    print(f"Train samples used for scaler: {scaler_fit_count}")
    print(f"This ensures NO data leakage from val/test into normalization")
else:
    if "split" not in metadata_df.columns:
        metadata_df["split"] = np.nan
    print("Feature engineer has no samples; skipped scaler/split export.")

Updated metadata with swing + quality fields → metadata.csv
✓ Splits assigned and saved to processed_videos\splits\dataset_splits.json

--- Fitting z-score scaler (train only, non-augmented) ---
✓ Scaler fit on 35 train samples
  Total sample records: 250
  Scaler stats count: 115500 feature vectors
✓ Scaler saved to features/feature_scaler.json
  Mean shape: (16,), Std shape: (16,)
✓ Normalized all saved sequences with train-only scaler
File locked, retrying in 1s... (attempt 1/3)
File locked, retrying in 1s... (attempt 2/3)
✓ Processed sample metadata: processed_videos\metadata\processed_metadata_new.csv
✓ Main metadata updated with split assignments

--- Scaler Fitting Summary ---
Train samples used for scaler: 35
This ensures NO data leakage from val/test into normalization


## Quick sanity checks
Review processed sample counts, low-quality ratio, and split distribution to validate Tasks G–H outcomes.

In [7]:
# --- Sanity summaries ---
if metadata_df.empty:
    print("Metadata frame is empty; nothing to summarize.")
else:
    print("Videos per env/band after processing:")
    band_summary = metadata_df.groupby(["env", "band"]).size().reset_index(name="video_count")
    display(band_summary)

    if "low_quality" in metadata_df.columns:
        print("\nLow-quality rate by environment:")
        quality_summary = (
            metadata_df.dropna(subset=["low_quality"])
            .groupby("env")["low_quality"]
            .mean()
            .reset_index(name="low_quality_rate")
        )
        display(quality_summary)
    else:
        print("\nNote: 'low_quality' column not found - quality stats skipped.")

    if "split" in metadata_df.columns:
        print("\nDataset split counts (by video_id):")
        split_summary = metadata_df["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="videos")
        display(split_summary)
        
        # Verify scaler was fit only on train
        if feature_engineer.samples:
            train_records = [r for r in feature_engineer.sample_records 
                           if r.get("split") == "train" and not r.get("augmented", False)]
            print(f"\n✓ Scaler Verification:")
            print(f"  Train (non-augmented) samples: {len(train_records)}")
            print(f"  These samples determined mean/std for normalization")
            print(f"  Val/test samples were normalized but did NOT influence scaler")
    else:
        print("\nNote: 'split' column not found - split stats skipped.")

    if error_log:
        print("\nWarnings (limited to 5):")
        for msg in error_log[:5]:
            print(" -", msg)
        if len(error_log) > 5:
            print(f"... {len(error_log) - 5} more")

Videos per env/band after processing:


,env,band,video_count
0,indoor,1_2,6
1,indoor,2_4,4
2,indoor,4_6,6
3,indoor,6_8,6
4,indoor,8_10,4
5,outdoor,1_2,4
6,outdoor,2_4,4
7,outdoor,4_6,6
8,outdoor,6_8,6
9,outdoor,8_10,4



Note: 'low_quality' column not found - quality stats skipped.

Dataset split counts (by video_id):


,split,videos
0,train,35
1,test,8
2,val,7



✓ Scaler Verification:
  Train (non-augmented) samples: 35
  These samples determined mean/std for normalization
  Val/test samples were normalized but did NOT influence scaler


## Pipeline Visualization Demo
Generate a side-by-side comparison video showing:
- **Left**: Original video with pose skeleton overlay
- **Right**: Normalized pose sequence (hip-centered, scaled)
- **Bottom**: Feature metrics timeline (wrist speed, X-factor, hip-shoulder separation)

In [8]:
# --- Visualization: Create demo video showing pipeline results ---
import cv2
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for rendering

# MediaPipe pose connections for skeleton drawing
POSE_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 7),  # Head
    (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10),  # Mouth
    (11, 12),  # Shoulders
    (11, 13), (13, 15),  # Left arm
    (12, 14), (14, 16),  # Right arm
    (11, 23), (12, 24),  # Torso
    (23, 24),  # Hips
    (23, 25), (25, 27),  # Left leg
    (24, 26), (26, 28),  # Right leg
]

def draw_skeleton(frame, landmarks, color=(0, 255, 0), thickness=2):
    """Draw pose skeleton on frame."""
    h, w = frame.shape[:2]
    points = []
    for i in range(33):
        x = int(landmarks[i, 0] * w)
        y = int(landmarks[i, 1] * h)
        vis = landmarks[i, 3] if landmarks.shape[1] > 3 else 1.0
        points.append((x, y, vis))
    
    # Draw connections
    for start, end in POSE_CONNECTIONS:
        if start < len(points) and end < len(points):
            if points[start][2] > 0.3 and points[end][2] > 0.3:
                cv2.line(frame, points[start][:2], points[end][:2], color, thickness)
    
    # Draw joints
    for x, y, vis in points:
        if vis > 0.3:
            cv2.circle(frame, (x, y), 4, (0, 0, 255), -1)
    
    return frame

def draw_normalized_skeleton(canvas_size, landmarks, color=(0, 255, 0), thickness=2):
    """Draw normalized pose on a blank canvas (centered at origin)."""
    h, w = canvas_size
    frame = np.zeros((h, w, 3), dtype=np.uint8)
    frame[:] = (30, 30, 30)  # Dark gray background
    
    # Scale and center
    scale = min(h, w) * 0.35
    cx, cy = w // 2, h // 2
    
    points = []
    for i in range(33):
        x = int(cx + landmarks[i, 0] * scale)
        y = int(cy - landmarks[i, 1] * scale)  # Flip Y for display
        vis = landmarks[i, 3] if landmarks.shape[1] > 3 else 1.0
        points.append((x, y, vis))
    
    # Draw connections
    for start, end in POSE_CONNECTIONS:
        if start < len(points) and end < len(points):
            if points[start][2] > 0.3 and points[end][2] > 0.3:
                cv2.line(frame, points[start][:2], points[end][:2], color, thickness)
    
    # Draw joints
    for x, y, vis in points:
        if vis > 0.3:
            cv2.circle(frame, (x, y), 5, (0, 165, 255), -1)  # Orange
    
    # Draw origin crosshair
    cv2.line(frame, (cx - 20, cy), (cx + 20, cy), (100, 100, 100), 1)
    cv2.line(frame, (cx, cy - 20), (cx, cy + 20), (100, 100, 100), 1)
    
    return frame

def create_metrics_plot(frame_idx, total_frames, features, canvas_size):
    """Create a plot showing feature metrics over time."""
    fig, axes = plt.subplots(3, 1, figsize=(canvas_size[1]/100, canvas_size[0]/100), dpi=100)
    fig.patch.set_facecolor('#1e1e1e')
    
    time_axis = np.arange(total_frames)
    current = frame_idx
    
    # Extract metrics (assuming features shape: T x 33 x 16)
    # Global metrics are replicated across joints, take from joint 0
    if len(features.shape) == 3:
        x_factor = features[:, 0, 13]  # X-factor
        hip_shoulder_sep = features[:, 0, 14]  # Hip-shoulder separation
        wrist_speed = features[:, 0, 15]  # Wrist speed
    else:
        x_factor = np.zeros(total_frames)
        hip_shoulder_sep = np.zeros(total_frames)
        wrist_speed = np.zeros(total_frames)
    
    metrics = [
        ("Wrist Speed", wrist_speed, '#00ff88'),
        ("X-Factor (°)", x_factor, '#ff6b6b'),
        ("Hip-Shoulder Sep", hip_shoulder_sep, '#4ecdc4'),
    ]
    
    for ax, (title, data, color) in zip(axes, metrics):
        ax.set_facecolor('#2d2d2d')
        ax.plot(time_axis, data, color=color, linewidth=1.5)
        ax.axvline(x=current, color='white', linewidth=2, alpha=0.8)
        ax.fill_between(time_axis[:current+1], 0, data[:current+1], alpha=0.3, color=color)
        ax.set_xlim(0, total_frames)
        ax.set_ylabel(title, color='white', fontsize=8)
        ax.tick_params(colors='white', labelsize=7)
        for spine in ax.spines.values():
            spine.set_color('#444444')
    
    axes[-1].set_xlabel('Frame', color='white', fontsize=8)
    plt.tight_layout()
    
    # Convert to image
    canvas = FigureCanvasAgg(fig)
    canvas.draw()
    buf = np.frombuffer(canvas.buffer_rgba(), dtype=np.uint8)
    buf = buf.reshape(fig.canvas.get_width_height()[::-1] + (4,))
    plt.close(fig)
    
    return cv2.cvtColor(buf, cv2.COLOR_RGBA2BGR)

def create_visualization_video(video_id, output_path=None):
    """Create a comprehensive visualization video for a processed sample."""
    
    # Find the video and its processed data
    video_row = metadata_df[metadata_df["video_id"] == video_id].iloc[0]
    video_path = Path(video_row["file_path"])
    if not video_path.is_absolute():
        video_path = config.BASE_DIR / video_path
    
    # Load processed features
    seq_path = config.OUTPUT_ROOT / config.FINAL_SEQUENCE_SUBDIR / f"{video_id}__{video_row['env']}__{video_row['band']}.npz"
    if not seq_path.exists():
        print(f"Processed sequence not found: {seq_path}")
        return None
    
    processed_data = np.load(seq_path, allow_pickle=True)
    features = processed_data["X"]  # Shape: (100, 33, 16)
    
    # Load original video
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Cannot open video: {video_path}")
        return None
    
    orig_fps = cap.get(cv2.CAP_PROP_FPS)
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Read all frames
    orig_frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        orig_frames.append(frame)
    cap.release()
    
    # Get swing window info
    swing_start = int(video_row.get("swing_start_frame", 0))
    swing_end = int(video_row.get("swing_end_frame", len(orig_frames)))
    
    # Setup output video
    panel_h, panel_w = 400, 400
    metrics_h = 200
    output_w = panel_w * 2
    output_h = panel_h + metrics_h
    
    if output_path is None:
        output_path = config.OUTPUT_ROOT / f"visualization_{video_id}.mp4"
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(output_path), fourcc, 30.0, (output_w, output_h))
    
    n_frames = features.shape[0]  # 100 frames (normalized)
    
    # Extract raw pose from features (first 4 channels: x, y, z, visibility)
    pose_coords = features[:, :, :4]
    
    print(f"Creating visualization for {video_id}...")
    print(f"  Original: {len(orig_frames)} frames @ {orig_fps:.1f} fps")
    print(f"  Swing window: frames {swing_start} → {swing_end}")
    print(f"  Normalized: {n_frames} frames")
    
    for i in tqdm(range(n_frames), desc="Rendering frames"):
        # Map normalized frame index to original video
        orig_idx = swing_start + int((swing_end - swing_start) * i / n_frames)
        orig_idx = min(orig_idx, len(orig_frames) - 1)
        
        # Left panel: Original video with skeleton
        orig_frame = orig_frames[orig_idx].copy()
        orig_frame = cv2.resize(orig_frame, (panel_w, panel_h))
        
        # Get pose for this frame (use normalized coordinates, scale to image)
        pose = pose_coords[i]
        # Denormalize from hip-centered to image coords (approximate)
        pose_img = pose.copy()
        pose_img[:, 0] = pose[:, 0] * 0.3 + 0.5  # Scale X
        pose_img[:, 1] = -pose[:, 1] * 0.3 + 0.5  # Scale Y (flip)
        orig_frame = draw_skeleton(orig_frame, pose_img, color=(0, 255, 0))
        
        # Add label
        cv2.putText(orig_frame, "Original + Pose", (10, 25), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(orig_frame, f"Frame {orig_idx}/{len(orig_frames)}", (10, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
        
        # Right panel: Normalized skeleton
        norm_frame = draw_normalized_skeleton((panel_h, panel_w), pose, color=(0, 255, 128))
        cv2.putText(norm_frame, "Normalized Pose", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(norm_frame, f"Frame {i+1}/{n_frames}", (10, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
        cv2.putText(norm_frame, "Hip-centered, scaled", (10, panel_h - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (150, 150, 150), 1)
        
        # Combine top panels
        top_row = np.hstack([orig_frame, norm_frame])
        
        # Bottom panel: Metrics timeline
        metrics_panel = create_metrics_plot(i, n_frames, features, (metrics_h, output_w))
        metrics_panel = cv2.resize(metrics_panel, (output_w, metrics_h))
        
        # Combine all
        final_frame = np.vstack([top_row, metrics_panel])
        
        out.write(final_frame)
    
    out.release()
    print(f"✓ Visualization saved to: {output_path}")
    return output_path

# Select a good quality video for demo (check column existence first)
if "low_quality" in metadata_df.columns:
    mask = (metadata_df["low_quality"] == False)
else:
    mask = pd.Series([True] * len(metadata_df))

if "processing_error" in metadata_df.columns:
    mask = mask & (metadata_df["processing_error"].isna())

good_samples = metadata_df[mask]

if len(good_samples) > 0:
    demo_video_id = good_samples.iloc[0]["video_id"]
    print(f"Selected demo video: {demo_video_id}")
else:
    # Fallback to first available
    demo_video_id = metadata_df.iloc[0]["video_id"]
    print(f"Using first available video: {demo_video_id}")

# Create the visualization
viz_path = create_visualization_video(demo_video_id)
if viz_path:
    print(f"\n🎬 Demo video created successfully!")
    print(f"   Open: {viz_path}")

Selected demo video: Backside-8767-14
Creating visualization for Backside-8767-14...
  Original: 325 frames @ 30.0 fps
  Swing window: frames 220 → 320
  Normalized: 100 frames


Rendering frames: 100%|██████████| 100/100 [00:09<00:00, 10.73it/s]

✓ Visualization saved to: D:\DAI_HOC\THI\DATASTORM 2025\v2\processed_videos\visualization_Backside-8767-14.mp4

🎬 Demo video created successfully!
   Open: D:\DAI_HOC\THI\DATASTORM 2025\v2\processed_videos\visualization_Backside-8767-14.mp4
